<a href="https://colab.research.google.com/github/PolinaBoeva/predictive_models_dota2/blob/models/experiments/Boeva/%D0%A7%D0%B5%D0%BA%D0%BF%D0%BE%D0%B8%D0%BD%D1%82_6_Dl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [172]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split

In [173]:
y_train = pd.read_csv('y_train.csv', index_col=0)
X_train = pd.read_csv('X_train.csv', index_col=0)
X_test = pd.read_csv('X_test.csv', index_col=0)
y_test = pd.read_csv('y_test.csv', index_col=0)

In [174]:
y_train_heroes = pd.read_csv('y_train_heroes.csv', index_col=0)
X_train_heroes = pd.read_csv('X_train_heroes.csv', index_col=0)
X_test_heroes = pd.read_csv('X_test_heroes.csv', index_col=0)
y_test_heroes = pd.read_csv('y_test_heroes.csv', index_col=0)

In [182]:
metrics_df = pd.DataFrame(columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"])

def get_model_metrics(y_test, predictions, X_test, model_name, predict_proba=None):
    global metrics_df

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)

    try:
        if predict_proba:
            proba = predict_proba(X_test)

            if isinstance(proba, pd.DataFrame):
                proba = proba.values

            if proba.ndim == 1:
                roc_auc = roc_auc_score(y_test, proba)
            elif proba.shape[1] == 1:
                roc_auc = roc_auc_score(y_test, proba.ravel())
            else:
                roc_auc = roc_auc_score(y_test, proba[:, 1])
        else:
            roc_auc = "N/A"
    except Exception as e:
        print("⚠️  Ошибка при вычислении ROC-AUC:", e)
        roc_auc = "N/A"

    print(f"\n📊 Metrics for {model_name}:")
    print(f"Accuracy:  {round(accuracy, 3)}")
    print(f"Precision: {round(precision, 3)}")
    print(f"Recall:    {round(recall, 3)}")
    print(f"F1 Score:  {round(f1, 3)}")
    print(f"ROC-AUC:   {round(roc_auc, 3) if roc_auc != 'N/A' else roc_auc}")

    new_metrics = pd.DataFrame([[model_name, round(accuracy, 3), round(precision, 3),
                                 round(recall, 3), round(f1, 3),
                                 round(roc_auc, 3) if roc_auc != "N/A" else roc_auc]],
                                columns=metrics_df.columns)

    metrics_df = pd.concat([metrics_df, new_metrics], ignore_index=True)

# MLP — Multi-Layer Perceptron

3 слоя: Linear -> ReLU -> BatchNorm -> Dropout

Выходной слой: Linear -> Sigmoid для бинарной классификации

Используется BCELoss (бинарная кросс-энтропия) — классическая функция потерь для задач "0 или 1".

Для решения задачи бинарной классификации была разработана и обучена с нуля полносвязная нейронная сеть (MLP) на фреймворке PyTorch. Архитектура включает несколько линейных слоёв с активацией ReLU, батч-нормализацией и дропаутами для борьбы с переобучением. Обучение модели производилось на масштабированных табличных данных с использованием функции потерь BCELoss и оптимизатора Adam. Выходной слой содержит сигмоиду, что позволяет получать вероятности классов для расчёта метрик, включая ROC-AUC.


In [194]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [195]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

In [196]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MLP(input_dim=X_train.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(100):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = criterion(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.6905
Epoch 2, Loss: 0.6315
Epoch 3, Loss: 0.6280
Epoch 4, Loss: 0.6947
Epoch 5, Loss: 0.7437
Epoch 6, Loss: 0.7080
Epoch 7, Loss: 0.7508
Epoch 8, Loss: 0.7157
Epoch 9, Loss: 0.6828
Epoch 10, Loss: 0.7448
Epoch 11, Loss: 0.7037
Epoch 12, Loss: 0.6574
Epoch 13, Loss: 0.7033
Epoch 14, Loss: 0.6338
Epoch 15, Loss: 0.6358
Epoch 16, Loss: 0.6182
Epoch 17, Loss: 0.6406
Epoch 18, Loss: 0.6651
Epoch 19, Loss: 0.6634
Epoch 20, Loss: 0.6585
Epoch 21, Loss: 0.5715
Epoch 22, Loss: 0.7034
Epoch 23, Loss: 0.6204
Epoch 24, Loss: 0.6079
Epoch 25, Loss: 0.5654
Epoch 26, Loss: 0.5451
Epoch 27, Loss: 0.5764
Epoch 28, Loss: 0.6426
Epoch 29, Loss: 0.6905
Epoch 30, Loss: 0.6373
Epoch 31, Loss: 0.7187
Epoch 32, Loss: 0.6054
Epoch 33, Loss: 0.5964
Epoch 34, Loss: 0.6047
Epoch 35, Loss: 0.5336
Epoch 36, Loss: 0.5431
Epoch 37, Loss: 0.6523
Epoch 38, Loss: 0.5827
Epoch 39, Loss: 0.5938
Epoch 40, Loss: 0.5353
Epoch 41, Loss: 0.6593
Epoch 42, Loss: 0.5034
Epoch 43, Loss: 0.5126
Epoch 44, Loss: 0.66

In [197]:
model.eval()
with torch.no_grad():
    logits = model(X_test_tensor.to(device)).cpu().numpy().flatten()
    y_pred_label = (logits > 0.5).astype(int)

def torch_predict_proba(X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32).to(device)).cpu().numpy()

get_model_metrics(
    y_test=y_test,
    predictions=y_pred_label,
    X_test=X_test_scaled,
    model_name="MLP-PyTorch",
    predict_proba=torch_predict_proba
)


📊 Metrics for MLP-PyTorch:
Accuracy:  0.568
Precision: 0.554
Recall:    0.771
F1 Score:  0.645
ROC-AUC:   0.606


# TabNet

Модель глубокого обучения, специально разработанная для табличных данных. Она использует механизм внимания для выбора наиболее информативных признаков на каждом шаге обработки данных. TabNet демонстрирует высокую производительность при меньших затратах на настройку гиперпараметров по сравнению с традиционными методами, такими как градиентный бустинг. Она также сохраняет интерпретируемость и эффективно избегает переобучения благодаря регуляризации. Подходит для задач классификации и регрессии на табличных данных.

In [ ]:
pip install pytorch-tabnet

In [192]:
from pytorch_tabnet.tab_model import TabNetClassifier

model = TabNetClassifier(
    n_d=64, n_a=64, n_steps=5, gamma=1.3, lambda_sparse=1e-5, optimizer_fn=torch.optim.Adam
)
model.fit(X_train=X_train_scaled, y_train=y_train.values.ravel(), max_epochs=100, patience=10, batch_size=1024, virtual_batch_size=128)

epoch 0  | loss: 0.9583  |  0:00:30s
epoch 1  | loss: 0.71139 |  0:00:49s
epoch 2  | loss: 0.69586 |  0:01:05s
epoch 3  | loss: 0.68217 |  0:01:23s
epoch 4  | loss: 0.68284 |  0:01:40s
epoch 5  | loss: 0.68238 |  0:01:57s
epoch 6  | loss: 0.6821  |  0:02:13s
epoch 7  | loss: 0.68191 |  0:02:30s
epoch 8  | loss: 0.68161 |  0:02:51s
epoch 9  | loss: 0.6824  |  0:03:17s
epoch 10 | loss: 0.68048 |  0:03:33s
epoch 11 | loss: 0.67995 |  0:03:54s
epoch 12 | loss: 0.67837 |  0:04:16s
epoch 13 | loss: 0.67831 |  0:04:34s
epoch 14 | loss: 0.6793  |  0:04:50s
epoch 15 | loss: 0.67652 |  0:05:07s
epoch 16 | loss: 0.67754 |  0:05:28s
epoch 17 | loss: 0.67843 |  0:05:44s
epoch 18 | loss: 0.67597 |  0:06:01s
epoch 19 | loss: 0.67502 |  0:06:17s
epoch 20 | loss: 0.67507 |  0:06:34s
epoch 21 | loss: 0.6755  |  0:06:50s
epoch 22 | loss: 0.67456 |  0:07:07s
epoch 23 | loss: 0.6751  |  0:07:24s
epoch 24 | loss: 0.67467 |  0:07:43s
epoch 25 | loss: 0.67451 |  0:08:00s
epoch 26 | loss: 0.67462 |  0:08:16s
e

In [193]:
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)

get_model_metrics(
    y_test=y_test,
    predictions=y_pred,
    X_test=X_test_scaled,
    model_name="TabNet",
    predict_proba=lambda X: y_pred_proba
)


📊 Metrics for TabNet:
Accuracy:  0.589
Precision: 0.59
Recall:    0.629
F1 Score:  0.609
ROC-AUC:   0.626


# Данные со статистикой по героям

# MLP — Multi-Layer Perceptron

In [199]:
scaler = StandardScaler()
X_train_heroes_scaled = scaler.fit_transform(X_train_heroes)
X_test_heroes_scaled = scaler.transform(X_test_heroes)

X_train_heroes_tensor = torch.tensor(X_train_heroes_scaled, dtype=torch.float32)
y_train_heroes_tensor = torch.tensor(y_train_heroes.values, dtype=torch.float32)

X_test_heroes_tensor = torch.tensor(X_test_heroes_scaled, dtype=torch.float32)
y_test_heroes_tensor = torch.tensor(y_test_heroes.values, dtype=torch.float32)

train_dataset = TensorDataset(X_train_heroes_tensor, y_train_heroes_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [200]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

In [202]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MLP(input_dim=X_train_heroes_tensor.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(100):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = criterion(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.6819
Epoch 2, Loss: 0.6662
Epoch 3, Loss: 0.6701
Epoch 4, Loss: 0.6696
Epoch 5, Loss: 0.6966
Epoch 6, Loss: 0.6644
Epoch 7, Loss: 0.5973
Epoch 8, Loss: 0.7323
Epoch 9, Loss: 0.6643
Epoch 10, Loss: 0.6771
Epoch 11, Loss: 0.6514
Epoch 12, Loss: 0.6316
Epoch 13, Loss: 0.6407
Epoch 14, Loss: 0.6794
Epoch 15, Loss: 0.6473
Epoch 16, Loss: 0.7027
Epoch 17, Loss: 0.6110
Epoch 18, Loss: 0.6292
Epoch 19, Loss: 0.6701
Epoch 20, Loss: 0.6548
Epoch 21, Loss: 0.4931
Epoch 22, Loss: 0.5969
Epoch 23, Loss: 0.7384
Epoch 24, Loss: 0.6794
Epoch 25, Loss: 0.5539
Epoch 26, Loss: 0.5566
Epoch 27, Loss: 0.5274
Epoch 28, Loss: 0.6968
Epoch 29, Loss: 0.6183
Epoch 30, Loss: 0.6139
Epoch 31, Loss: 0.5527
Epoch 32, Loss: 0.5891
Epoch 33, Loss: 0.6037
Epoch 34, Loss: 0.5975
Epoch 35, Loss: 0.6358
Epoch 36, Loss: 0.5588
Epoch 37, Loss: 0.5821
Epoch 38, Loss: 0.5946
Epoch 39, Loss: 0.6050
Epoch 40, Loss: 0.5622
Epoch 41, Loss: 0.5805
Epoch 42, Loss: 0.6209
Epoch 43, Loss: 0.4659
Epoch 44, Loss: 0.66

In [207]:
model.eval()
with torch.no_grad():
    logits = model(X_test_heroes_tensor.to(device)).cpu().numpy().flatten()
    y_pred_label = (logits > 0.5).astype(int)

def torch_predict_proba(X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32).to(device)).cpu().numpy()

get_model_metrics(
    y_test=y_test_heroes,
    predictions=y_pred_label,
    X_test=X_test_heroes_scaled,
    model_name="MLP-PyTorch with heroes",
    predict_proba=torch_predict_proba
)


📊 Metrics for MLP-PyTorch with heroes:
Accuracy:  0.575
Precision: 0.614
Recall:    0.443
F1 Score:  0.514
ROC-AUC:   0.618


# TabNet

In [208]:
from pytorch_tabnet.tab_model import TabNetClassifier

model = TabNetClassifier(
    n_d=64, n_a=64, n_steps=5, gamma=1.3, lambda_sparse=1e-5, optimizer_fn=torch.optim.Adam
)
model.fit(X_train=X_train_heroes_scaled, y_train=y_train_heroes.values.ravel(), max_epochs=100, patience=10, batch_size=1024, virtual_batch_size=128)

epoch 0  | loss: 0.91793 |  0:00:24s
epoch 1  | loss: 0.70506 |  0:00:42s
epoch 2  | loss: 0.68704 |  0:01:02s
epoch 3  | loss: 0.69001 |  0:01:20s
epoch 4  | loss: 0.68508 |  0:01:39s
epoch 5  | loss: 0.68446 |  0:01:58s
epoch 6  | loss: 0.68391 |  0:02:17s
epoch 7  | loss: 0.68082 |  0:02:36s
epoch 8  | loss: 0.68153 |  0:02:54s
epoch 9  | loss: 0.67929 |  0:03:13s
epoch 10 | loss: 0.68018 |  0:03:32s
epoch 11 | loss: 0.67781 |  0:03:52s
epoch 12 | loss: 0.67811 |  0:04:11s
epoch 13 | loss: 0.67651 |  0:04:30s
epoch 14 | loss: 0.67678 |  0:04:50s
epoch 15 | loss: 0.67731 |  0:05:09s
epoch 16 | loss: 0.67974 |  0:05:27s
epoch 17 | loss: 0.67887 |  0:05:47s
epoch 18 | loss: 0.67884 |  0:06:06s
epoch 19 | loss: 0.67767 |  0:06:26s
epoch 20 | loss: 0.67601 |  0:06:44s
epoch 21 | loss: 0.67617 |  0:07:03s
epoch 22 | loss: 0.67616 |  0:07:23s
epoch 23 | loss: 0.6749  |  0:07:41s
epoch 24 | loss: 0.67355 |  0:08:00s
epoch 25 | loss: 0.67299 |  0:08:19s
epoch 26 | loss: 0.6737  |  0:08:37s
e

In [212]:
y_pred = model.predict(X_test_heroes_scaled)
y_pred_proba = model.predict_proba(X_test_heroes_scaled)

get_model_metrics(
    y_test=y_test_heroes,
    predictions=y_pred,
    X_test=X_test_heroes_scaled,
    model_name="TabNet with heroes",
    predict_proba=lambda X: y_pred_proba
)


📊 Metrics for TabNet with heroes:
Accuracy:  0.578
Precision: 0.577
Recall:    0.642
F1 Score:  0.607
ROC-AUC:   0.617


In [215]:
metrics_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
1,TabNet,0.589,0.590,0.629,0.609,0.626
2,MLP-PyTorch,0.568,0.554,0.771,0.645,0.606
5,MLP-PyTorch with heroes,0.575,0.614,0.443,0.514,0.618
8,TabNet with heroes,0.578,0.577,0.642,0.607,0.617
